In [ ]:
cd ..

In [ ]:
from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
from src.ingestion.last_activity_generator import generate_last_activity



In [ ]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()
players_silver = spark.read.parquet("./data/silver/players")
sessions_silver = spark.read.parquet("./data/silver/sessions")
transactions_silver = spark.read.parquet("./data/silver/transactions")
silver_money_events = spark.read.parquet("./data/silver/money_events")


In [ ]:

players_silver = players_silver.drop('player_id')
transactions_silver = transactions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
sessions_silver = sessions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
silver_money_events = silver_money_events.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')


In [ ]:
first_last_activity = generate_last_activity(silver_money_events)
first_last_activity = (
    first_last_activity
    .join(
        players_silver.select("player_idx"),
        on="player_idx",
        how="inner"
    )
)


In [ ]:
player_snapshot = (players_silver
                   .select('player_idx','country','age_bucket','device_type','acquisition_channel', 'registration_date', 'risk_segment', 'lifecycle_stage', F.col('current_balance').alias('current_balance'))
                   .join(first_last_activity,
                         on='player_idx',
                         how='left')
                         .withColumn('days_since_last_session', 
                                     F.date_diff(F.lit(config_.end_date), F.col('last_session_date')))
                         .drop('reference_date')

)
player_snapshot.show(5)

## Construct training dataset for ML

### Full dataframe (players x dates)

In [ ]:
date_range = spark.sql(
    f"SELECT explode(sequence(to_date('{config_.start_date}'), to_date('{config_.end_date}'), interval 1 day)) AS event_date"
)

df_pl_date = (player_snapshot
              .select('player_idx', 'registration_date','first_event_date', 'last_event_date',  'first_session_date',  'last_session_date', 'first_financial_date', 'last_financial_date')
              .crossJoin(date_range)
            .filter(F.col('first_session_date').isNotNull())  #new players
            .withColumn('session_date_days', 
                F.datediff(F.col("event_date"), F.lit("1970-01-01")))
)
window_7d = (Window.partitionBy('player_idx').orderBy('session_date_days').rangeBetween(-6,0))
window_30d = (Window.partitionBy('player_idx').orderBy('session_date_days').rangeBetween(-29,0))

### Compute rolling futures based on sessions

In [ ]:
df_sessions_detail = (df_pl_date.select('player_idx','event_date','registration_date', 'first_session_date','last_session_date','session_date_days')
.join(sessions_silver
      .select('session_id',
 'session_duration_sec',
 'bet_count',
 'total_bet_amount',
 'total_win_amount',
 'signed_amount',
 'balance_after_txn',
   F.col('session_date').alias('event_date'), 
   'player_idx'),
        how='left', on=['player_idx','event_date'])
.filter( F.col("first_session_date")<= F.col("event_date"))
#.filter(F.datediff(F.col("event_date"), F.col("first_session_date")) > 30)
#.filter(F.datediff(F.lit(config_.end_date), F.col("event_date")) > 7)
.orderBy('player_idx', 'event_date' )
)

In [ ]:
df_sessions_rolling = (df_sessions_detail
            .withColumn('num_sessions_7d', F.count('session_id').over(window_7d))
            .withColumn('net_game_result_7d', F.sum('signed_amount').over(window_7d))
            .withColumn('num_sessions_30d', F.count('session_id').over(window_30d))
            .withColumn('net_game_result_30d', F.sum('signed_amount').over(window_30d))  
            .withColumn('avg_sessions_duration_30d', F.avg('session_duration_sec').over(window_30d))
            .withColumn('avg_bet_amount_30d', F.avg('total_bet_amount').over(window_30d))
                       ) 
(df_sessions_rolling.filter(F.datediff(F.col("event_date"), F.col("first_session_date")) > 30)
.filter(F.datediff(F.lit(config_.end_date), F.col("event_date")) > 7)).show()

### Compute rolling futures based on all events

In [ ]:
silver_money_events_detail =  (df_pl_date.select('player_idx','event_date','registration_date', 'first_event_date','last_event_date','session_date_days')
.join(silver_money_events
      .select('event_id',
 'event_type',
 'signed_amount',
 'balance_after_txn',
   F.col('event_ts').alias('event_date'), 
   'player_idx'),
        how='left', on=['player_idx','event_date'])
.filter( F.col("first_event_date")<= F.col("event_date"))
#.filter(F.datediff(F.col("event_date"), F.col("first_session_date")) > 30)
#.filter(F.datediff(F.lit(config_.end_date), F.col("event_date")) > 7)
.orderBy('player_idx', 'event_date' )
)
silver_money_events_detail.show(4)

In [ ]:
window_up_to_7d = (Window.partitionBy('player_idx').orderBy('session_date_days').rangeBetween(Window.unboundedPreceding,-6))
window_up_to_30d = (Window.partitionBy('player_idx').orderBy('session_date_days').rangeBetween(Window.unboundedPreceding,-29))

silver_money_events_rolling = (silver_money_events_detail
            .withColumn('balance_7d_ago', F.last('balance_after_txn', ignorenulls=True).over(window_up_to_7d))
            .withColumn('net_amount_result_7d', F.sum('signed_amount').over(window_7d))
            .withColumn('balance_30d_ago', F.last('balance_after_txn', ignorenulls=True).over(window_up_to_7d))
            .withColumn('net_amount_result_30d', F.sum('signed_amount').over(window_30d))  
                       ) 
(silver_money_events_rolling.filter(F.datediff(F.col("event_date"), F.col("first_event_date")) > 30)
.filter(F.datediff(F.lit(config_.end_date), F.col("event_date")) > 7)).show()

### Compute rolling futures based on financial transactions 

In [ ]:
transactions_detail = (df_pl_date.select('player_idx','event_date','registration_date', 'first_financial_date','last_financial_date','session_date_days')
.join(transactions_silver
      .select('transaction_id',
 'transaction_type',
 'amount',
 'success_flag',
 'signed_amount',
 'balance_after_txn',
   F.to_date(F.col('transaction_ts')).alias('event_date'), 
   'player_idx'),
        how='left', on=['player_idx','event_date'])
.filter( F.col("first_financial_date")<= F.col("event_date"))
#.filter(F.datediff(F.col("event_date"), F.col("first_session_date")) > 30)
#.filter(F.datediff(F.lit(config_.end_date), F.col("event_date")) > 7)
.orderBy('player_idx', 'event_date' )
)


In [ ]:

transactions_rolling = (transactions_detail
                .withColumn('failed_withdrawals_30d', 
                    F.sum(F.when((F.col('success_flag')==False) & (F.col('transaction_type')=='withdrawal'), 1).otherwise(0)).over(window_30d))
                .withColumn('deposit_count_30d', 
                    F.sum(F.when(F.col('transaction_type')=='deposit',1).otherwise(0)).over(window_30d))
                .withColumn('withdrawal_count_30d', 
                    F.sum(F.when(F.col('transaction_type')=='withdrawal',1).otherwise(0)).over(window_30d))
                .withColumn('withdrawal_ratio',
                     F.when(
                        F.col("deposit_count_30d") > 0,
                        F.col("withdrawal_count_30d") / F.col("deposit_count_30d")
                         ).otherwise(F.lit(0.0)))
)


### Compose the final training dataset

In [ ]:
gold_player_behavior = (silver_money_events_rolling
                        .join(df_sessions_rolling, how='left', on='player_idx') 
                        .join(transactions_rolling, how='left', on='player_idx') 

)
gold_player_behavior.select('player_idx',
 'net_amount_result_7d',
 'net_amount_result_30d',
 'failed_withdrawals_30d',
 'deposit_count_30d',
 'withdrawal_count_30d',
 'withdrawal_ratio',
 'num_sessions_7d',
 'net_game_result_7d',
 'num_sessions_30d',
 'avg_sessions_duration_30d',
 'avg_bet_amount_30d',
 'net_game_result_30d',
 'balance_7d_ago',
 'balance_30d_ago').show(5)

### Create gold labels

In [ ]:
df_num_of_sessions = (sessions_silver
.groupBy('player_idx', 'session_date')
.agg(F.count('*').alias('num_of_session')))
#.orderBy('player_idx', 'session_date' ))

df_sessions = (df_pl_date
.join(df_num_of_sessions
      .select('num_of_session', F.col('session_date').alias('event_date'), 'player_idx'),
        how='left', on=['player_idx','event_date'])
)

sessions_silver_sec = (df_sessions
                       .filter(F.col('event_date') >= F.col('first_session_date'))
            .withColumn('num_sessions_7d', 
                        F.sum(F.when(F.col("num_of_session") > 0, 1).otherwise(0)).over(window_7d))
            .withColumn('churn_7d', F.when(F.col('num_sessions_7d')==0, True).otherwise(False))
            .withColumn('num_sessions_30d', 
                        F.sum(F.when(F.col('num_of_session') > 0, 1).otherwise(0)).over(window_30d))
           .withColumn('churn_30d', F.when(F.col('num_sessions_30d')==0, True).otherwise(False))
)

In [ ]:
window_7d_ahead = (Window.partitionBy('player_idx').orderBy('session_date_days').rangeBetween(1,6))
target = (sessions_silver_sec
            .withColumn('next_7d_churn', 
            (F.max(F.col("churn_7d").cast("int")).over(window_7d_ahead) == 1))
)

## Player_behavior based on the last day

In [ ]:
silver_money_events = silver_money_events.withColumn('days_diff', F.date_diff(F.lit(config_.end_date), F.col('event_ts')))
sessions_silver = sessions_silver.withColumn('days_diff', F.date_diff(F.lit(config_.end_date), F.col('session_date')))
transactions_silver = transactions_silver.withColumn('days_diff', F.date_diff(F.lit(config_.end_date),F.to_date(F.col('transaction_ts'))))

### player_behavior based on sessions activity

In [ ]:
player_behavior = (sessions_silver.withColumn('days_diff', F.date_diff(F.lit(config_.end_date), F.col('session_date')))
 .filter(F.col('days_diff') <= 30)
 .groupBy('player_idx')
 .agg(
            F.sum(F.when(F.col('days_diff') <= 7, 1).otherwise(0)).alias('sessions_7d'),
            F.sum(F.when(F.col('days_diff') <= 7, F.col('signed_amount')).otherwise(0)).alias('net_game_result_7d'),

            F.count('*').alias('sesions_30d'),
            F.avg(F.col('session_duration_sec')).cast('int').alias('avg_session_duration_sec_30d'),
            F.avg(F.col('total_bet_amount')).alias('avg_bet_amount_30d'),
            F.sum(F.col('signed_amount')).alias('net_game_result_30d'),
 )
                
)
player_behavior.show()

### player behavior based on any money event (financial + bet)

In [ ]:

### Correct ###
silver_money_events_net = (silver_money_events
 .filter(F.col('days_diff') <= 30)
 .groupBy('player_idx')
 .agg(
            F.sum(F.when(F.col('days_diff') <= 7, F.col('signed_amount')).otherwise(0)).alias('net_amount_result_7d'),
            F.sum(F.col('signed_amount')).alias('net_amount_result_30d'),
 )        
)

silver_money_events_net.show()

In [ ]:
w = Window.partitionBy("player_idx").orderBy(F.col("event_ts").desc())
player_30d = (silver_money_events
 .filter(F.col('days_diff') >=30)
 .withColumn('rn', F.row_number().over(w))
.filter(F.col('rn') ==1)
)
player_30d_act= (players_silver
                   .select('player_idx','current_balance')
                   .join(player_30d.select('player_idx','balance_after_txn'),
                         on='player_idx',
                         how='left')
                             .withColumn(
                              "balance_30d_ago",
                              F.coalesce("balance_after_txn", "current_balance"))
                              .drop( 'current_balance', 'balance_after_txn')
)
player_30d_act.show()

In [ ]:
w = Window.partitionBy("player_idx").orderBy(F.col("event_ts").desc())
player_7d = (silver_money_events
 .filter(F.col('days_diff') >=7)
 .withColumn('rn', F.row_number().over(w))
.filter(F.col('rn') ==1)
)
player_7d_act= (players_silver
                   .select('player_idx','current_balance')
                   .join(player_7d.select('player_idx',F.col('balance_after_txn')),
                         on='player_idx',
                         how='left')
                             .withColumn(
                              "balance_7d_ago",
                              F.coalesce("balance_after_txn", "current_balance"))
                              .drop('current_balance', 'balance_after_txn')
)
player_7d_act.show()

In [ ]:
transaction_behavior = (transactions_silver
.filter(F.col('days_diff')<=30)
.groupBy(F.col('player_idx'))
.agg(
    F.sum(F.when((F.col('transaction_type')=='withdrawal') & (F.col('success_flag')==False),1).otherwise(0)).alias('failed_withdrawals_30d'),
    F.sum(F.when(F.col('transaction_type')=='deposit',1).otherwise(0)).alias('deposit_count_30d'),
    F.sum(F.when(F.col('transaction_type')=='withdrawal',1).otherwise(0)).alias('withdrawal_count_30d'),
)
.withColumn(        'withdrawal_ratio',
    F.when(
            F.col("deposit_count_30d") > 0,
            F.col("withdrawal_count_30d") / F.col("deposit_count_30d")
        ).otherwise(F.lit(0.0))
)
)
transaction_behavior.show()

In [ ]:
gold_player_behavior = (players_silver.select('player_idx')
                        .join(silver_money_events_net, how='left', on='player_idx') 
                        .join(transaction_behavior, how='left', on='player_idx') 
                        .join(player_behavior, how='left', on='player_idx') 
                        .join(player_7d_act, how='left', on='player_idx') 
                        .join(player_30d_act, how='left', on='player_idx') 

)
gold_player_behavior.show()

In [ ]:
gold_player_behavior.columns

In [ ]:
sessions_silver.filter(F.col('player_idx')==2).orderBy('session_date').show()

# Examples 

In [ ]:
sessions_silver_sec.filter(F.col('churn_7d')==True).filter(F.col('event_date')>'2024-01-07') .orderBy('event_date').show()

In [ ]:
target.filter(F.col('churn_7d')).filter(F.col('next_7d_churn')==False).show()

In [ ]:
df_sessions.filter(F.col('player_idx')==10059).show()
